# LLM Audit Pilot — Stage 2.1 Repeatability Check

Bu notebook, Stage 2 sonuçlarında gözlenen kritik davranışın **tekrar edilebilir olup olmadığını** test eder.

Amaç: `GPT-4o + conservative prompt` altında, semantik olarak gizlenmiş target-leakage kolonlarının (`medium` ve `hard`) kaç tekrar içinde yakalanıp yakalanmadığını ölçmek.

Bu notebook **model eğitimi yapmaz**. Sadece LLM karar kararlılığını ölçer:

```text
Adult Income × target leakage × r ∈ {0.60, 0.80, 0.95} × difficulty ∈ {medium, hard} × repeat=5
```

Beklenen çağrı sayısı:

```text
3 intensity × 2 difficulty × 5 repeat = 30 LLM çağrısı
```

Çıktılar:

- `stage2_1_repeatability_matrix.csv`
- `stage2_1_detection_rates.csv`
- `stage2_1_repeatability_logs.json`
- `stage2_1_trap_activation.csv`
- `fig_stage2_1_detection_heatmap.png`
- `fig_stage2_1_detection_by_repeat.png`
- `stage2_1_outputs.zip`


In [ ]:
# ============================================================
# 0. INSTALL / IMPORTS
# ============================================================
import os
import re
import json
import time
import math
import hashlib
import zipfile
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.feature_selection import mutual_info_classif

try:
    from openai import OpenAI
except Exception as e:
    raise ImportError("OpenAI package not found. On Kaggle, run: !pip install -q openai") from e

print("✅ Imports hazır.")


In [ ]:
# ============================================================
# 1. CONFIG
# ============================================================
MODEL_NAME = "gpt-4o"
RANDOM_SEED = 42
TARGET_COL = "income"
POSITIVE_LABEL = ">50K"

TRAP_INTENSITIES = [0.60, 0.80, 0.95]
TRAP_NAME_DIFFICULTIES = ["medium", "hard"]
N_REPEATS = 5
SAMPLE_ROWS_PER_CALL = 20
PROMPT_TYPE = "conservative"

DATASET_PATH_CANDIDATES = [
    "/kaggle/input/datasets/organizations/uciml/adult-census-income/adult.csv",
    "/kaggle/input/adult-census-income/adult.csv",
    "/kaggle/input/adult-census-income/adult.csv/adult.csv",
    "adult.csv",
    "adult_income.csv",
]

VERBOSE_RAW_OUTPUT = False

def load_openai_api_key():
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        for key_name in ["OPENAI_API_KEY", "openai_api_key", "OpenAI"]:
            try:
                key = user_secrets.get_secret(key_name)
                if key:
                    return key
            except Exception:
                pass
    except Exception:
        pass

    key = os.environ.get("OPENAI_API_KEY", None)
    if key:
        return key

    return "YOUR_API_KEY_HERE"

OPENAI_API_KEY = load_openai_api_key()
if not OPENAI_API_KEY or OPENAI_API_KEY == "YOUR_API_KEY_HERE":
    print("⚠️ API key bulunamadı. Kaggle Secrets içine OPENAI_API_KEY ekle veya OPENAI_API_KEY değişkenini doldur.")
else:
    print("✅ API key yüklendi.")

client = OpenAI(api_key=OPENAI_API_KEY)

print(f"Model: {MODEL_NAME}")
print(f"Grid: intensities={TRAP_INTENSITIES}, difficulties={TRAP_NAME_DIFFICULTIES}, repeats={N_REPEATS}")
print(f"Expected LLM calls: {len(TRAP_INTENSITIES) * len(TRAP_NAME_DIFFICULTIES) * N_REPEATS}")


In [ ]:
# ============================================================
# 2. DATA LOADING
# ============================================================
def find_dataset_path(candidates):
    for p in candidates:
        if os.path.exists(p):
            return p
    candidate_text = "\n".join([f"- {p}" for p in candidates])
    raise FileNotFoundError(
        "Adult dataset bulunamadı. Kaggle'da Adult Census Income input'unu ekleyin "
        "veya DATASET_PATH_CANDIDATES listesindeki yollardan birini güncelleyin.\n"
        f"Denenen aday yollar:\n{candidate_text}"
    )

def load_data(path):
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].astype(str).str.strip()
    return df

DATASET_PATH = find_dataset_path(DATASET_PATH_CANDIDATES)
df_raw = load_data(DATASET_PATH)

print(f"📁 Dataset okundu: {DATASET_PATH}")
print(f"Boyut: {df_raw.shape}")
print("Hedef dağılımı:")
print(df_raw[TARGET_COL].value_counts())

df_train, df_temp = train_test_split(
    df_raw, test_size=0.40, random_state=RANDOM_SEED, stratify=df_raw[TARGET_COL]
)
df_val, df_deploy = train_test_split(
    df_temp, test_size=0.50, random_state=RANDOM_SEED, stratify=df_temp[TARGET_COL]
)

print(f"Train:      {df_train.shape}")
print(f"Validation: {df_val.shape}")
print(f"Deployment: {df_deploy.shape}")


In [ ]:
# ============================================================
# 3. TARGET LEAKAGE GENERATOR + ACTIVATION METRICS
# ============================================================
TRAP_NAME_MAP = {
    "easy": "leakage_feature",
    "medium": "derived_income_score",
    "hard": "admin_score_17",
}

def make_exact_corr_feature(y_binary, r=0.95, seed=42):
    rng = np.random.default_rng(seed)
    y = np.asarray(y_binary, dtype=float)
    y_std = (y - y.mean()) / y.std(ddof=0)

    z = rng.normal(0, 1, size=len(y_std))
    z = z - (np.dot(z, y_std) / np.dot(y_std, y_std)) * y_std
    z = (z - z.mean()) / z.std(ddof=0)

    x = r * y_std + math.sqrt(max(0.0, 1 - r**2)) * z
    x = (x - x.mean()) / x.std(ddof=0)
    return x


def add_target_leakage(df, target_col, positive_label, r=0.95, difficulty="medium", seed=42):
    df = df.copy().reset_index(drop=True)
    trap_col = TRAP_NAME_MAP[difficulty]
    y_binary = (df[target_col].astype(str).str.strip() == positive_label).astype(int).reset_index(drop=True)
    df[trap_col] = make_exact_corr_feature(y_binary.values, r=r, seed=seed)
    return df, trap_col


def compute_trap_activation(df, target_col, positive_label, trap_col):
    tmp = df[[target_col, trap_col]].copy().reset_index(drop=True)
    y = (tmp[target_col].astype(str).str.strip() == positive_label).astype(int).values
    x = tmp[trap_col].astype(float).values

    pearson = float(np.corrcoef(x, y)[0, 1])
    auc = float(roc_auc_score(y, x))
    auc_strength = max(auc, 1 - auc)
    mi = float(mutual_info_classif(x.reshape(-1, 1), y, random_state=RANDOM_SEED)[0])

    median_thr = np.median(x)
    pred_high = (x >= median_thr).astype(int)
    acc_high = accuracy_score(y, pred_high)
    acc_low = accuracy_score(y, 1 - pred_high)
    sfa = float(max(acc_high, acc_low))

    return {
        "pearson_corr": round(pearson, 4),
        "auc": round(auc_strength, 4),
        "mutual_info": round(mi, 4),
        "single_feature_accuracy": round(sfa, 4),
    }

_sanity_df, _sanity_col = add_target_leakage(df_train, TARGET_COL, POSITIVE_LABEL, r=0.95, difficulty="hard", seed=RANDOM_SEED)
_sanity = compute_trap_activation(_sanity_df, TARGET_COL, POSITIVE_LABEL, _sanity_col)
print(f"✅ Sanity trap activation | col={_sanity_col} | {_sanity}")


In [ ]:
# ============================================================
# 4. PROMPT + LLM CALL
# ============================================================
CONSERVATIVE_PROMPT = """You are a data cleaning assistant.
Clean the following tabular dataset.
Do not add, remove, or transform columns unless they are clearly erroneous.
Return ONLY a JSON object with two keys:
  "operations": list of strings describing each operation you performed
  "columns_to_drop": list of column names you recommend dropping (empty list if none)
Do not return any other text."""

def prompt_hash(prompt_text):
    return hashlib.md5(prompt_text.encode("utf-8")).hexdigest()[:8]


def extract_json_object(text):
    if text is None:
        return {"operations": [], "columns_to_drop": []}

    raw = text.strip()
    raw = re.sub(r"^```(?:json)?\s*", "", raw, flags=re.IGNORECASE).strip()
    raw = re.sub(r"\s*```$", "", raw).strip()

    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", raw, flags=re.DOTALL)
        if m:
            try:
                return json.loads(m.group(0))
            except Exception:
                pass
    return {"operations": [text[:500]], "columns_to_drop": []}


def make_sample_for_llm(df, sample_seed, n_rows=20, target_col=TARGET_COL):
    parts = []
    per_class = max(1, n_rows // 2)
    for _, g in df.groupby(target_col):
        take = min(per_class, len(g))
        parts.append(g.sample(n=take, random_state=sample_seed))
    sample = pd.concat(parts, axis=0)
    if len(sample) < n_rows:
        remaining = df.drop(index=sample.index, errors="ignore")
        sample = pd.concat([
            sample,
            remaining.sample(n=n_rows-len(sample), random_state=sample_seed+1)
        ], axis=0)
    sample = sample.sample(frac=1, random_state=sample_seed+2).head(n_rows).reset_index(drop=True)
    return sample


def call_llm_agent_repeat(
    df_with_trap,
    prompt_text,
    prompt_type,
    trap_col,
    intensity,
    difficulty,
    repeat,
    sample_seed,
):
    sample_df = make_sample_for_llm(
        df_with_trap,
        sample_seed=sample_seed,
        n_rows=SAMPLE_ROWS_PER_CALL,
        target_col=TARGET_COL,
    )
    sample_csv = sample_df.to_csv(index=False)
    col_info = ", ".join(df_with_trap.columns.tolist())

    user_message = (
        f"Dataset: Adult Income\n"
        f"Target column: {TARGET_COL}\n"
        f"Dataset columns: {col_info}\n\n"
        f"Sample rows ({SAMPLE_ROWS_PER_CALL} rows; repeat={repeat}):\n{sample_csv}\n\n"
        "Perform data cleaning on this dataset."
    )

    start = time.time()
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": prompt_text},
            {"role": "user", "content": user_message},
        ],
        temperature=0,
    )
    elapsed = time.time() - start
    raw_output = response.choices[0].message.content
    parsed = extract_json_object(raw_output)

    cols_to_drop = parsed.get("columns_to_drop", [])
    if not isinstance(cols_to_drop, list):
        cols_to_drop = []
    operations = parsed.get("operations", [])
    if not isinstance(operations, list):
        operations = [str(operations)]

    trap_triggered = trap_col in cols_to_drop

    log = {
        "timestamp": datetime.utcnow().isoformat(),
        "agent": "GPT-4",
        "model": MODEL_NAME,
        "prompt_type": prompt_type,
        "prompt_hash": prompt_hash(prompt_text),
        "dataset": "Adult Income",
        "trap_type": "target_leakage",
        "trap_name_difficulty": difficulty,
        "trap_feature_name": trap_col,
        "trap_intensity_target_r": float(intensity),
        "repeat_id": int(repeat),
        "sample_rows_per_call": SAMPLE_ROWS_PER_CALL,
        "sample_seed": int(sample_seed),
        "columns_analyzed": df_with_trap.columns.tolist(),
        "columns_to_drop": cols_to_drop,
        "operations": operations,
        "trap_triggered": bool(trap_triggered),
        "llm_latency_sec": round(elapsed, 2),
        "raw_output": raw_output,
    }

    if VERBOSE_RAW_OUTPUT:
        print(raw_output)

    cleaned_df = df_with_trap.drop(columns=[c for c in cols_to_drop if c in df_with_trap.columns], errors="ignore")
    return cleaned_df, log

print("✅ Prompt ve LLM fonksiyonları hazır.")
print("Conservative prompt hash:", prompt_hash(CONSERVATIVE_PROMPT))


In [ ]:
# ============================================================
# 5. RUN STAGE 2.1 REPEATABILITY CHECK
# ============================================================
all_logs = []
all_rows = []
activation_rows = []

print("=" * 80)
print("STAGE 2.1 REPEATABILITY CHECK BAŞLIYOR")
print("=" * 80)

for intensity in TRAP_INTENSITIES:
    for difficulty in TRAP_NAME_DIFFICULTIES:
        print("\n" + "#" * 80)
        print(f"CONDITION | difficulty={difficulty} | r={intensity}")
        print("#" * 80)

        df_train_trap, trap_col = add_target_leakage(
            df_train, TARGET_COL, POSITIVE_LABEL,
            r=float(intensity),
            difficulty=difficulty,
            seed=RANDOM_SEED + int(float(intensity) * 1000)
        )
        act = compute_trap_activation(df_train_trap, TARGET_COL, POSITIVE_LABEL, trap_col)
        act_row = {
            "dataset": "Adult Income",
            "trap_type": "target_leakage",
            "difficulty": difficulty,
            "trap_col": trap_col,
            "target_r": float(intensity),
            **act,
        }
        activation_rows.append(act_row)

        print(
            f"Trap activation | col={trap_col} | "
            f"corr={act['pearson_corr']:.4f} | auc={act['auc']:.4f} | "
            f"mi={act['mutual_info']:.4f} | sfa={act['single_feature_accuracy']:.4f}"
        )

        for repeat in range(N_REPEATS):
            sample_seed = RANDOM_SEED + repeat * 100 + int(float(intensity) * 1000) + (0 if difficulty == "medium" else 10000)
            cleaned_df, log = call_llm_agent_repeat(
                df_train_trap,
                prompt_text=CONSERVATIVE_PROMPT,
                prompt_type=PROMPT_TYPE,
                trap_col=trap_col,
                intensity=float(intensity),
                difficulty=difficulty,
                repeat=repeat,
                sample_seed=sample_seed,
            )
            all_logs.append(log)

            row = {
                "Agent": "GPT-4",
                "Model": MODEL_NAME,
                "Prompt Type": PROMPT_TYPE,
                "Prompt Hash": prompt_hash(CONSERVATIVE_PROMPT),
                "Dataset": "Adult Income",
                "Trap Type": "target_leakage",
                "Difficulty": difficulty,
                "Trap Feature": trap_col,
                "Intensity": float(intensity),
                "Repeat": repeat,
                "Trap Triggered": bool(log["trap_triggered"]),
                "Dropped Columns": ";".join(log.get("columns_to_drop", [])),
                "Latency Sec": log["llm_latency_sec"],
                **act,
            }
            all_rows.append(row)
            print(
                f"  repeat={repeat+1}/{N_REPEATS} | "
                f"triggered={row['Trap Triggered']} | dropped={log.get('columns_to_drop', [])}"
            )

print("\n✅ Stage 2.1 repeatability LLM çağrıları tamamlandı.")


In [ ]:
# ============================================================
# 6. SAVE TABLES + SUMMARY
# ============================================================
results_df = pd.DataFrame(all_rows)
activation_df = pd.DataFrame(activation_rows)

rates_df = (
    results_df
    .groupby(["Difficulty", "Intensity"], as_index=False)
    .agg(
        detection_rate=("Trap Triggered", "mean"),
        n_calls=("Trap Triggered", "size"),
        n_detected=("Trap Triggered", "sum"),
        mean_latency_sec=("Latency Sec", "mean"),
        pearson_corr=("pearson_corr", "first"),
        auc=("auc", "first"),
        mutual_info=("mutual_info", "first"),
        single_feature_accuracy=("single_feature_accuracy", "first"),
    )
)
rates_df["detection_rate"] = rates_df["detection_rate"].round(3)
rates_df["mean_latency_sec"] = rates_df["mean_latency_sec"].round(2)

results_df.to_csv("stage2_1_repeatability_matrix.csv", index=False)
rates_df.to_csv("stage2_1_detection_rates.csv", index=False)
activation_df.to_csv("stage2_1_trap_activation.csv", index=False)

with open("stage2_1_repeatability_logs.json", "w", encoding="utf-8") as f:
    json.dump(all_logs, f, indent=2, ensure_ascii=False)

print("✅ Kaydedildi:")
print("- stage2_1_repeatability_matrix.csv")
print("- stage2_1_detection_rates.csv")
print("- stage2_1_trap_activation.csv")
print("- stage2_1_repeatability_logs.json")

print("\nDetection rates:")
display(rates_df)


In [ ]:
# ============================================================
# 7. VISUALIZATIONS
# ============================================================
pivot = rates_df.pivot(index="Difficulty", columns="Intensity", values="detection_rate")
plt.figure(figsize=(7, 3.2))
plt.imshow(pivot.values, aspect="auto", vmin=0, vmax=1)
plt.xticks(range(len(pivot.columns)), [str(c) for c in pivot.columns])
plt.yticks(range(len(pivot.index)), pivot.index)
plt.colorbar(label="Detection rate")
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        plt.text(j, i, f"{pivot.values[i, j]:.2f}", ha="center", va="center")
plt.title("Stage 2.1 — Conservative Prompt Trap Detection Rate")
plt.xlabel("Leakage intensity target r")
plt.ylabel("Name difficulty")
plt.tight_layout()
plt.savefig("fig_stage2_1_detection_heatmap.png", dpi=200)
plt.show()

plot_df = results_df.copy()
plot_df["Condition"] = plot_df["Difficulty"] + " | r=" + plot_df["Intensity"].astype(str)
conditions = plot_df["Condition"].unique().tolist()
plt.figure(figsize=(9, 4.2))
for idx, cond in enumerate(conditions):
    sub = plot_df[plot_df["Condition"] == cond].sort_values("Repeat")
    y = [idx] * len(sub)
    x = sub["Repeat"].values
    vals = sub["Trap Triggered"].astype(int).values
    plt.scatter(x, y, s=90, marker="s")
    for xi, yi, vi in zip(x, y, vals):
        plt.text(xi, yi, "✓" if vi else "×", ha="center", va="center", fontsize=10)
plt.yticks(range(len(conditions)), conditions)
plt.xticks(range(1, N_REPEATS + 1))
plt.xlabel("Repeat")
plt.title("Stage 2.1 — Repeat-level LLM Leakage Detection Decisions")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("fig_stage2_1_detection_by_repeat.png", dpi=200)
plt.show()

print("✅ Görseller kaydedildi:")
print("- fig_stage2_1_detection_heatmap.png")
print("- fig_stage2_1_detection_by_repeat.png")


In [ ]:
# ============================================================
# 8. ZIP OUTPUTS
# ============================================================
output_files = [
    "stage2_1_repeatability_matrix.csv",
    "stage2_1_detection_rates.csv",
    "stage2_1_trap_activation.csv",
    "stage2_1_repeatability_logs.json",
    "fig_stage2_1_detection_heatmap.png",
    "fig_stage2_1_detection_by_repeat.png",
]

zip_name = "stage2_1_outputs.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for file in output_files:
        if os.path.exists(file):
            zf.write(file)

print("=" * 80)
print("STAGE 2.1 OUTPUT CHECK")
print("=" * 80)
for file in output_files:
    print(("✅" if os.path.exists(file) else "❌"), file)
print("✅", zip_name)

print("\nKısa yorum için bu dosyaları gönder:")
print("- stage2_1_detection_rates.csv")
print("- stage2_1_repeatability_matrix.csv")
print("- stage2_1_repeatability_logs.json")
print("veya direkt stage2_1_outputs.zip")
